This notebook demonstrates how to run the forest deforestation User Defined Process

In [13]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [14]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [15]:
connection.authenticate_oidc()

INFO:openeo.rest.connection:Found OIDC providers: ['CDSE']
INFO:openeo.rest.connection:No OIDC provider given, but only one available: 'CDSE'. Using that one.
INFO:openeo.rest.connection:Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
INFO:openeo.rest.connection:Found refresh token: trying refresh token based authentication.
INFO:openeo.rest.auth.oidc:Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
INFO:openeo.rest.connection:Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
INFO:openeo.rest.auth.config:Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

# Parameters

In [16]:
spatial_extent = {
    "west": 32.637423,
    "south": 0.060209,
    "east": 33.131945,
    "north": 0.291267,
}

# ascending relative orbit that intersects this box. 101 does not.
relative_orbit = 28

In [17]:
# minimum canopy cover to be considered forest
# should be one of (10, 20, 30, 40, 50, 60, 70, 80, 90)
# a value of 30 means > 30% canopy cover
canopy_cover_threshold = 30

# minimum likelihood to be considered natural forest
natural_forest_threshold = 0.08

# minimum connected area to be considered forest (m^2)
min_connected_area = 10000

In [18]:
spatial_resolution = 30  # m

In [20]:
temporal_variability_threshold = 0.6
flattening_threshold = 0.18
logistic_sse_threshold = 80

In [21]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
# temporal_variability_threshold = 0.6
# flattening_threshold = 0.14

In [22]:
cropland_probability_threshold = 0.1

# Script

In [24]:
kpis_vector_cube = connection.datacube_from_process(
    "End-to-end forest-loss KPIs",
    namespace=urls.END_TO_END_UDP,
    spatial_extent=spatial_extent,
    relative_orbit=relative_orbit,
    canopy_cover_threshold=canopy_cover_threshold,
    natural_forest_threshold=natural_forest_threshold,
    min_connected_area=min_connected_area,
    spatial_resolution=spatial_resolution,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
    logistic_sse_threshold=logistic_sse_threshold,
    cropland_probability_threshold=cropland_probability_threshold,
)

# Batch job

In [25]:
job = kpis_vector_cube.create_job(out_format="Parquet")

In [26]:
job.start_and_wait()
# 1:11, 139 credits

0:00:00 Job 'j-2611231622524be6ba8340782c7585a6': send 'start'
0:00:16 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:00:22 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:00:28 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:00:36 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:00:46 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:00:59 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:01:15 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:01:34 Job 'j-2611231622524be6ba8340782c7585a6': queued (progress 0%)
0:01:58 Job 'j-2611231622524be6ba8340782c7585a6': running (progress N/A)
0:02:28 Job 'j-2611231622524be6ba8340782c7585a6': running (progress N/A)
0:03:06 Job 'j-2611231622524be6ba8340782c7585a6': running (progress N/A)
0:03:52 Job 'j-2611231622524be6ba8340782c7585a6': running (progress N/A)
0:04:51 Job 'j-2611231622524be6ba8340782c7585a6': running (progress N/A)
0:05

<BatchJob job_id='j-2611231622524be6ba8340782c7585a6'>

In [27]:
results = job.get_results()

In [28]:
!mkdir -p output-udp/
!rm -r output-udp/

In [29]:
results.download_files("output-udp/")

INFO:openeo.rest.job:Downloading job result asset 'timeseries.parquet' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-2611231622524be6ba8340782c7585a6/timeseries.parquet?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=e3e84b917df84050bb52f859f6e9e782%2F20261123%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20261123T164348Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYxMTIzMTYyMjUyNGJlNmJhODM0MDc4MmM3NTg1YTYiXSwidXNlcl9pZCI6WyIxM2YyYTg1Mi00YzFjLTQ4N2EtOGExYS0xYzA1YjE2M2I4YTIiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZCJdfSwiaXNzIjoic3RzLndhdzMtMS5vcGVuZW8udjEuZGF0YXNwYWNl

[PosixPath('output-udp/timeseries.parquet'),
 PosixPath('output-udp/job-results.json')]

In [30]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)